# Rust (1987) Coding and Estimation
## Rust setting
The paper models the decision-making process of Harold Zucher, who makes replacement choices for buses each period. Buses age and engines deteriorate over time, motivating HZ to replace the engine at a cost. You don't want to replace too early, and you don't want to replace too late. 

The choice to replace features uncertainty from the perspective of me, the econometrician. In other words, HZ observes something we don't, and this actually allows the model to be estimated -- otherwise the model is easily rejected by data -- it cannot explain why a replacement would occur at 10000 miles but not 12000 miles. 

### State
The state here is simply the milage of the bus (engine): 

$$
s_t = \{ 1, ..., 10\}
$$

State transitions: with probability $\lambda$ the mileage of the bus increases in the next period

$$
s_{t+1} = \begin{cases}
\min \{s_t + 1, 10\} & \text{ with probablitity } \lambda \\ 
s_t & \text{ with probability } 1-\lambda
\end{cases} 
$$

### Actions
Action available to HZ is a replacement decision 
$$
a_t = \{ 0,1\}
$$

Payoffs: 
Two payoffs, conditional on actions. If no replacement, has per-period maintenance. If replacement, has cost of replacement

$$
u(s_t, a_t, \varepsilon_{0t}, \varepsilon_{1t};\theta) = \begin{cases}
-\theta_1 s_t -\theta_2 s_t^2 + \varepsilon_{0t} & \text{ if } a_t = 0 \\ 
-\theta_3 + \varepsilon_{1t} & \text{ if } a_t =1 
\end{cases}
$$

## Solving the model
1. Start with an initial expected value function $V(s_t)=0$
2. Compute the alternative-specific value function: 
$$
\bar V(s_t) = \begin{cases}
-\theta_1 s_t -\theta_2 s_t^2 + \beta \left[ (1-\lambda)V(s_t)+\lambda V(\min \{ s_t + 1, 10 \}) \right] & \text{ if } a_t=0 \\ 
-\theta_3 + \beta\left[ (1-\lambda)V(0)+\lambda V(1) \right] & \text{ if } a_t=1
\end{cases}
$$

3. Compute the new expected value function: 

$$
V'(s_t) = \log \left( \exp(\bar V(s_t | a_t=0) +  \exp (\bar V(s_t | a_t=1) )\right) 
$$

## Code
We now move to actually estimating the model (and generating data). 

In [1]:
import numpy as np
import pandas as pd
# Set parameters
theta = [0.13, -0.004, 3.1]
lam = 0.82 # cannot use lambda directly in python
beta = 0.95

# Set the state space
k = 10
s = np.arange(k)


### Static Utility
Can compute directly from 
$$
u(s_t, a_t, \varepsilon_{0t}, \varepsilon_{1t};\theta) = \begin{cases}
-\theta_1 s_t -\theta_2 s_t^2 + \varepsilon_{0t} & \text{ if } a_t = 1 \\ 
-\theta_3 + \varepsilon_{1t} & \text{ if } a_t =1 
\end{cases}
$$

In [2]:
def compute_u(theta, s): 
    ''' Compute static utility'''
    u1 = -theta[0]* s -theta[1] * s**2
    u2 = -theta[2] * np.ones_like(s)

    U = np.column_stack((u1, u2))
    return U

### Value Function Iteration
Set it up!

In [3]:
def compute_vbar(theta, lam, beta, s):
    ''' 
    Compute value function by Bellman iteration
    '''
    k = len(s) # Dimension of state space
    U = compute_u(theta, s) # static utility
    # Mileage transition index
    #   col 0: stay at current state  -> 0,1,...,k-1
    #   col 1: advance one state       -> 1,2,...,k-1,k-1 (capped at top)
    index_lam = np.column_stack((
        np.arange(k),
        np.append(np.arange(1, k), k - 1),
    ))

    # Action index
    #   col 0: keep    -> continue from current state
    #   col 1: replace -> reset to state 0 (engine as good as new)
    index_A = np.column_stack((
        np.arange(k),
        np.zeros(k, dtype=int),
    ))
    gamma = np.euler_gamma

    # Iterate Bellman until convergence to the unique fixed point
    vbar = np.zeros((k, 2))
    dist = 1
    iterations = 0
    while dist > 1e-8:
        v = gamma + np.log(np.sum(np.exp(vbar), axis = 1))
        expv = v[index_lam] @ np.array([1-lam, lam])
        vbar1 = U + beta *expv[index_A]

        dist = np.max(np.abs(vbar1 - vbar))
        iterations += 1
        vbar = vbar1
    return vbar

Can now solve the model!

In [4]:
V_bar = compute_vbar(theta, lam, beta, s)
V_bar

array([[4.20997766, 1.10997766],
       [3.69383086, 1.10997766],
       [3.26288799, 1.10997766],
       [2.90361337, 1.10997766],
       [2.60424856, 1.10997766],
       [2.35546854, 1.10997766],
       [2.15079703, 1.10997766],
       [1.98716246, 1.10997766],
       [1.86608325, 1.10997766],
       [1.79602251, 1.10997766]])

### Data Generating Process
Now need to simulate some data


In [5]:
np.random.seed(1987)
def generate_data(theta, lam, beta, s, N):
    vbar = compute_vbar(theta, lam, beta, s) # Solve model
    eps = np.random.gumbel(0, 1, size = (N, 2)) # draw shocks
    st = np.random.choice(s, size= N) # draw current states
    # compute investment decision
    util = vbar[st, :] + eps 
    A = ((util @ np.array([-1,1])) > 0).astype(int)

    delta = (np.random.uniform(0,1, size = N)< lam).astype(int)

    st1 = np.minimum(st * (A==0) + delta, s.max())


    df = pd.DataFrame({"State": st, "Action": A, "State1": st1})
    df.to_csv("../rust.csv", index = False)
    return st, A, st1

### Generate the data
We can now generate the data using the various functions defined above. 

In [6]:
# Gen data
N = 100000
st, A, st1 = generate_data(theta, lam, beta, s, N)

print(f"We observe {sum(A)} investment decisions in {N} observations")

We observe 19903 investment decisions in 100000 observations


### Data
Let's view it to see what is actually looks like:

In [7]:
df = pd.read_csv('../rust.csv')
df.head()

,State,Action,State1
0,3,0,3
1,5,0,6
2,7,1,1
3,9,0,9
4,2,0,3


# Estimate $\lambda$
Estimate $\lambda$ -- the probability of mileage increase, conditional on not replacing the engine, and not being in the last state where mileage can no longer increase

$$
\hat \lambda = \mathbb{E}_N \Big[ (s_{t+1}-s_t) \mid a_t =0 \wedge s_t <10 \Big]
$$

Here, $\mathbb{E}_N$ represents the empirical expectation, which is simply a sample average over the $N$ data points. Effectively, we just take an average over $s_{t+1}-s_t$ over the subset of data points where the conditions on the RHS are met. Specifically, engine was kept $a_t=0$ and the mileage was not already at the maximum $s_t<10$. 

Note that the mileages here have been discretized. It is not 10 miles, but rather the 10th bin. 


In [8]:
# del lambda
df['del_lam'] = df['State1'] - df['State']     # brackets, not df.del_lam

mask = (df['Action'] == 0) & (df['State'] < df['State'].max())
est_lam = df.loc[mask, 'del_lam'].mean()

print(f"We set lambda={lam}, and the model estimates hat lambda= {est_lam}")

We set lambda=0.82, and the model estimates hat lambda= 0.8202070913084405


### Estimation of $\theta$
1. We have to take an initial guess of parameters $\theta_0$
2. Compute the alternative-specific value function $\bar V(s_t;\hat \lambda, \theta_0)$ by VFI
3. Compute the impied choice probabilities
4. Compute the likelihood: 
$$
L(\theta) = \Pi_{t=1}^T \Big( \hat{Pr}(a=1|s_t, \theta) \mathbb{1}\{a_t=1 \} + \left( 1-\hat{Pr}(a=0 | s_t, \theta) \right) \mathbb{1}\{a_t=0\} \Big)
$$
5. Repeat the above to find a minimum of the likelihood function.

### Likelihood function
We use this to estimate $\theta$. We know the 'true' value of $\theta$ from our code used in the DGP: 

$$
\theta = [\theta_1, \theta_2, \theta_3] = [0.13, -0.004, 3.1]
$$

Obviously when we simply have data and do not generate it, we do not know the true values as we do here. So, we will make our initial guess of $\theta$ and denote it $\theta_0$. 

In [9]:
def log_likelihood_rust(theta0, lam, beta, s, St, A):
    '''
    Compute log likelihood function for Rust
    '''
    # compute value functions
    vbar = compute_vbar(theta0, lam, beta, s)

    # Expected choice probabilities
    ep = np.exp(vbar[:, 1]) / (np.exp(vbar[:, 0]) + np.exp(vbar[:, 1])) 

    # Log likelihood
    logL = np.sum(np.log(ep[St[A==1]])) + np.sum(np.log(1-ep[St[A==0]]))
    return -logL 

In [10]:
# Check likelihood with true parameter vector
logL_true = log_likelihood_rust(theta, lam, beta, s, st, A)

print(f"The likelihood at the true parameter is {logL_true:.2f}. This does not mean much at first glance, so let's make it more interpretable: dividing by the number of observations gives {-logL_true / N:.4f}. We can then compute the average probability assigned to the realized choice: {np.exp(-logL_true / N):.4f}")

The likelihood at the true parameter is 46458.77. This does not mean much at first glance, so let's make it more interpretable: dividing by the number of observations gives -0.4646. We can then compute the average probability assigned to the realized choice: 0.6284


The likelihood we computed can be used to map to probabilities. We find that the model assigns, on average, about 0.63 probability to the choice Harold Zucher actually made at each bus. In slightly different words, 0.63 is the average probability that the model assigns to the choice that actually happened. I should note that this is not an equivalent statement to 'The model is 63% accurate'. 

### Estimating parameters
In reality, we do not know the true parameters $\theta$. If we did, we would not need to be doing any of this. 

In [11]:
# Take a guess of parameter values -- these are our initial/starting values

from scipy.optimize import minimize 
# The minimize tool imported above will search for an input that minimizes
# the function we feed it. 
theta0 = [0, 0, 0]

result = minimize(
    lambda x: log_likelihood_rust(x, lam, beta, s, st, A),
    theta0, 
    method = 'Nelder-Mead' # default optimizing method in Julia
)

theta_R = result.x 

print(f'Estimated thetas: {theta_R} (true = {theta})')

Estimated thetas: [ 0.13147467 -0.00412011  3.09863219] (true = [0.13, -0.004, 3.1])


### Summary

In this project, we developed a working Python implementation of the Rust (1987) model. We began with a set of parameters $\boldsymbol{\theta}$ which we then used to simulate data.We recovered these parameters in two steps: the transition probability $\lambda$ as a direct conditional sample average, and the cost parameters $\hat{\boldsymbol{\theta}}$ by maximum likleihood, where each uniquie likelihood evaluation solves HZ's dynamic problem for its unique fixed point via value function iteration.  With sensible starting values, our optimizer recovered estimates  $\hat{\boldsymbol{\theta}}$ close to the true parameters. 

### References
Rust, John. 1988. “Maximum Likelihood Estimation of Discrete Control Processes.” SIAM Journal on Control and Optimization 26 (5): 1006–24.

Based on the Julia code of Matteo Courthood: https://matteocourthoud.github.io/course/empirical-io/17_rust_1987/

### Notes
Note the optimizer chosen (method) is Nelder-Mead. This is the default optimizer used in Julia -- the language of choice of Matteo Courthood, from which this code is adapted. 

We could try an alternative method to minimize the function, such as BFGS. However, BFGS requires a gradient, which this problem lacks. Let's try it regardless: 

In [ ]:
result = minimize(
    lambda x: log_likelihood_rust(x, lam, beta, s, st, A),
    theta0, 
    method = 'BFGS' 
)

theta_R = result.x 

print(f'Estimated thetas: {theta_R} (true = {theta})')

C:\Users\Daniel\AppData\Local\Temp\ipykernel_28560\1565076102.py:29: RuntimeWarning: overflow encountered in exp
  v = gamma + np.log(np.sum(np.exp(vbar), axis = 1))
C:\Users\Daniel\AppData\Local\Temp\ipykernel_28560\1565076102.py:33: RuntimeWarning: invalid value encountered in subtract
  dist = np.max(np.abs(vbar1 - vbar))
C:\Users\Daniel\AppData\Local\Temp\ipykernel_28560\2021162834.py:12: RuntimeWarning: divide by zero encountered in log
  logL = np.sum(np.log(ep[St[A==1]])) + np.sum(np.log(1-ep[St[A==0]]))
c:\Users\Daniel\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\optimize\_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0
C:\Users\Daniel\AppData\Local\Temp\ipykernel_28560\1565076102.py:29: RuntimeWarning: overflow encountered in exp
  v = gamma + np.log(np.sum(np.exp(vbar), axis = 1))
C:\Users\Daniel\AppData\Local\Temp\ipykernel_28560\1565076102.py:33: RuntimeWarning: invalid value encountered in subtract
  dist = np.ma

Estimated thetas: [ 0.13147761 -0.00412043  3.09863343] (true = [0.13, -0.004, 3.1])


The BFGS method does quite well, although it throws us some warnings since it's not well-suited to the problem. 

### Initial guess matters
Picking good starting values matters for estimation: 

In [27]:
theta0 = [1, 1, 1]

result = minimize(
    lambda x: log_likelihood_rust(x, lam, beta, s, st, A),
    theta0, 
    method = 'Nelder-Mead' 
)

theta_R = result.x 

print(f'Estimated thetas: {np.round(theta_R, 2)} (true = {theta})')

C:\Users\Daniel\AppData\Local\Temp\ipykernel_5932\2021162834.py:12: RuntimeWarning: divide by zero encountered in log
  logL = np.sum(np.log(ep[St[A==1]])) + np.sum(np.log(1-ep[St[A==0]]))
c:\Users\Daniel\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\optimize\_optimize.py:851: RuntimeWarning: invalid value encountered in subtract
  np.max(np.abs(fsim[0] - fsim[1:])) <= fatol):


Estimated thetas: [1. 1. 1.] (true = [0.13, -0.004, 3.1])
